# Tutorial 09 — Glider Wing Optimisation (Aero-Only)

This tutorial shows how to use the `Opti` inline API to declare wing design variables
and run an aero-only optimisation on a simple unpowered glider.

**What you'll learn:**
- Build a vanilla glider (no motor, no battery)
- Declare design variables inline with `opti.variable()` — no string paths required
- Scope the discipline chain to aero only (`disciplines=["aero"]`)
- Maximise lift-to-drag ratio subject to a minimum-CL constraint
- Inspect baseline vs optimal aerodynamics and interpret the sensitivity report

**Design variables** (both on the wing tip section):

| Variable | Baseline | Range |
|---|---|---|
| Tip chord | 0.14 m | 0.07 – 0.26 m |
| Tip twist (washout) | −2 deg | −6 – +1 deg |

**Objective:** maximise `CL / CD` (lift-to-drag ratio)

**Constraint:** `CL ≥ 0.4` — keep the wing in a reasonable lift regime

In [1]:
import warnings
import logging

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import aerisplane as ap
from aerisplane.mdo import Opti, Objective, Constraint

logging.basicConfig(level=logging.WARNING)
warnings.filterwarnings('ignore')

## 1. Define the glider

The glider has three lifting surfaces: a tapered main wing, a horizontal tail, and
a vertical fin. There is no propulsion system — this is a pure sailplane geometry.

The two design variables are declared **inline** using `opti.variable()` when setting
the tip section chord and twist. AerisPlane walks the aircraft dataclass at
problem-creation time to discover these automatically — no dot-bracket path strings needed.

In [2]:
ag35     = ap.Airfoil(name='ag35')          # thin cambered: good low-Re glider airfoil
naca0009 = ap.Airfoil.from_naca('0009')    # symmetric: tail surfaces

opti = Opti()  # context object that tracks inline design variables

# ── Main wing ─────────────────────────────────────────────────────────────────
# Root section is fixed. Tip chord and twist are design variables.
main_wing = ap.Wing(
    name='main_wing',
    xsecs=[
        ap.WingXSec(
            xyz_le=[0.0, 0.00, 0.00],
            chord=0.28,          # root chord fixed at 280 mm
            twist=0.0,           # root twist fixed
            airfoil=ag35,
        ),
        ap.WingXSec(
            xyz_le=[0.04, 1.20, 0.04],   # 1.2 m semi-span, 4 cm sweep, 4 cm dihedral
            chord=opti.variable(0.14, lower=0.07, upper=0.26),  # tip chord [m]
            twist=opti.variable(-2.0, lower=-6.0, upper=1.0),   # washout [deg]
            airfoil=ag35,
        ),
    ],
    symmetric=True,
)

# ── Horizontal tail ───────────────────────────────────────────────────────────
htail = ap.Wing(
    name='htail',
    xsecs=[
        ap.WingXSec(xyz_le=[0.85, 0.00, 0.00], chord=0.12, airfoil=naca0009),
        ap.WingXSec(xyz_le=[0.87, 0.28, 0.00], chord=0.09, airfoil=naca0009),
    ],
    symmetric=True,
)

# ── Vertical fin ──────────────────────────────────────────────────────────────
vtail = ap.Wing(
    name='vtail',
    xsecs=[
        ap.WingXSec(xyz_le=[0.82, 0.0, 0.00], chord=0.16, airfoil=naca0009),
        ap.WingXSec(xyz_le=[0.86, 0.0, 0.22], chord=0.10, airfoil=naca0009),
    ],
    symmetric=False,
)

# ── Assemble ──────────────────────────────────────────────────────────────────
aircraft = ap.Aircraft(
    name='glider',
    wings=[main_wing, htail, vtail],
    fuselages=[],
)

print('Baseline main wing geometry:')
print(f'  Span:        {main_wing.span():.2f} m')
print(f'  Area:        {main_wing.area():.4f} m\u00b2')
print(f'  AR:          {main_wing.aspect_ratio():.2f}')
print(f'  Taper ratio: {main_wing.taper_ratio():.2f}')
print(f'  Root chord:  {main_wing.xsecs[0].chord:.3f} m')
print(f'  Tip chord:   {main_wing.xsecs[1].chord:.3f} m  \u2190 design variable')
print(f'  Tip twist:   {main_wing.xsecs[1].twist:.1f} deg  \u2190 design variable')

Baseline main wing geometry:
  Span:        2.40 m
  Area:        0.5040 m²
  AR:          11.43
  Taper ratio: 0.50
  Root chord:  0.280 m
  Tip chord:   0.140 m  ← design variable
  Tip twist:   -2.0 deg  ← design variable


**Expected output:**
```
Baseline main wing geometry:
  Span:        2.40 m
  Area:        0.5040 m²
  AR:          11.43
  Taper ratio: 0.50
  Root chord:  0.280 m
  Tip chord:   0.140 m  ← design variable
  Tip twist:   -2.0 deg  ← design variable
```

## 2. Inspect baseline aerodynamics

Before optimising, run `aero_buildup` directly so we can see what the baseline wing
looks like. `aero_buildup` includes both induced drag (CDi from VLM) and profile drag
(CDp from NeuralFoil section polars), giving a realistic L/D estimate.

We fix **α = 4 deg** — a typical cruise angle of attack for this type of glider.

In [3]:
from aerisplane.aero import analyze

cruise = ap.FlightCondition(velocity=12.0, altitude=300.0, alpha=4.0)

# Quick aero check at baseline — use the raw aircraft (before opti replaces tip values)
baseline_aircraft = ap.Aircraft(
    name='glider_baseline',
    wings=[
        ap.Wing(name='main_wing', xsecs=[
            ap.WingXSec(xyz_le=[0.0, 0.00, 0.00], chord=0.28, twist=0.0,  airfoil=ag35),
            ap.WingXSec(xyz_le=[0.04, 1.20, 0.04], chord=0.14, twist=-2.0, airfoil=ag35),
        ], symmetric=True),
        htail,
        vtail,
    ],
    fuselages=[],
)

r_base = analyze(baseline_aircraft, cruise, method='aero_buildup', model_size='xxsmall')

print('Baseline aerodynamics at \u03b1 = 4 deg, V = 12 m/s, alt = 300 m:')
print(f'  CL   = {r_base.CL:.4f}')
print(f'  CD   = {r_base.CD:.4f}  (CDi={r_base.CDi:.4f}, CDp={r_base.CD - r_base.CDi:.4f})')
print(f'  L/D  = {r_base.CL_over_CD:.2f}')

Baseline aerodynamics at α = 4 deg, V = 12 m/s, alt = 300 m:
  CL   = 0.7748
  CD   = 0.0360  (CDi=0.0204, CDp=0.0156)
  L/D  = 21.51


**Expected output:**
```
Baseline aerodynamics at α = 4 deg, V = 12 m/s, alt = 300 m:
  CL   = 0.7748
  CD   = 0.0360  (CDi=0.0204, CDp=0.0156)
  L/D  = 21.51
```

The profile drag (CDp) accounts for almost half the total drag at this Reynolds number
(Re ≈ 190 000). Reducing chord reduces wetted area and CDp, which is why the optimizer
will prefer a more slender tip.

## 3. Set up the optimisation problem

`opti.problem()` walks the `aircraft` dataclass tree, finds all `opti.variable()` tags,
and builds an `MDOProblem` automatically. We scope the run to `disciplines=["aero"]`
so weights, structures, and stability are skipped — the evaluation loop is fast.

> **Note:** `opti.variable()` only works for scalar `float` fields on dataclasses.
> Fields stored inside a list or NumPy array (like `xyz_le`) must still use the
> `DesignVar("wings[0].xsecs[1].xyz_le[1]", ...)` string-path API.

In [4]:
problem = opti.problem(
    aircraft=aircraft,
    condition=cruise,
    disciplines=['aero'],
    objective=Objective('aero.CL_over_CD', maximize=True),
    constraints=[Constraint('aero.CL', lower=0.4)],
    alpha=4.0,               # fix angle of attack — no trim solver needed
    aero_method='aero_buildup',
)

print(f'Design variables discovered: {len(problem._dvars)}')
for dv in problem._dvars:
    val = problem._baseline
    print(f'  {dv.path:<45}  [{dv.lower:.3f}, {dv.upper:.3f}]')

print(f'\nConstraints: {[c.path for c in problem._constraints]}')
print(f'Disciplines: {problem._disciplines}')

Design variables discovered: 2
  wings[0].xsecs[1].chord                        [0.070, 0.260]
  wings[0].xsecs[1].twist                        [-6.000, 1.000]

Constraints: ['aero.CL']
Disciplines: ['weights', 'aero']


**Expected output:**
```
Design variables discovered: 2
  wings[0].xsecs[1].chord                           [0.070, 0.260]
  wings[0].xsecs[1].twist                           [-6.000, 1.000]

Constraints: ['aero.CL']
Disciplines: ['weights', 'aero']
```

`weights` is always included — it sets the CG reference point before the aero run.
Both paths were auto-discovered from the `opti.variable()` tags in the wing definition.

## 4. Run the baseline simulation

`problem.simulate()` evaluates the discipline chain once at the baseline design.
Useful for sanity-checking before committing to a full optimizer run.

In [5]:
baseline = problem.simulate()

r0 = baseline['aero']
print('Baseline (via MDOProblem):')
print(f'  CL   = {r0.CL:.4f}')
print(f'  CD   = {r0.CD:.4f}')
print(f'  L/D  = {r0.CL_over_CD:.2f}')
print()

# Check constraint at baseline
x0 = problem._x0_scaled()
violations = problem.constraint_functions(x0)
print(f'CL constraint violation at baseline: {violations[0]:.4f}  (\u22640 = satisfied)')

Baseline (via MDOProblem):
  CL   = 0.7623
  CD   = 0.0345
  L/D  = 22.10

CL constraint violation at baseline: -0.3623  (≤0 = satisfied)


**Expected output:**
```
Baseline (via MDOProblem):
  CL   = 0.7623
  CD   = 0.0345
  L/D  = 22.10

CL constraint violation at baseline: -0.3623  (≤0 = satisfied)
```

The constraint violation is negative (satisfied): CL = 0.76 is well above the 0.4 lower bound.

## 5. Optimise

Differential evolution (`scipy_de`) is a gradient-free global optimizer — ideal for
aero problems that may have a slightly non-smooth landscape.

With only 2 variables and fast aero evaluations, `maxiter=40, popsize=10` converges
quickly. We disable polishing (`polish=False`) to avoid SciPy's gradient-based
refinement step, which can fail if the aero response has small discontinuities.

In [6]:
result = problem.optimize(
    method='scipy_de',
    options={'maxiter': 40, 'seed': 42, 'popsize': 10, 'polish': False},
    verbose=False,
)

print(result.report())

AerisPlane Optimisation Result

Objective
----------------------------------------
  Initial  : -22.1021
  Optimal  : -23.3953
  Change   : -5.9%
  Evals    : 82
  Constraints satisfied: YES

Design Variables
----------------------------------------
  Path                                             Initial    Optimal
  --------------------------------------------- ---------- ----------
  wings[0].xsecs[1].chord                             0.14   0.091904
  wings[0].xsecs[1].twist                               -2    -5.9205


**Expected output:**
```
AerisPlane Optimisation Result
============================================================

Objective
----------------------------------------
  Initial  : -22.1021
  Optimal  : -23.3991
  Change   : -5.9%
  Evals    : 82
  Constraints satisfied: YES

Design Variables
----------------------------------------
  Path                                             Initial    Optimal
  --------------------------------------------- ---------- ----------
  wings[0].xsecs[1].chord                             0.14   0.098475
  wings[0].xsecs[1].twist                               -2     -5.876
```

The objective sign is negative because L/D is being **maximised** (stored internally
as a minimisation problem). The actual improvement is L/D: 22.1 → 23.4 (+5.9%).

The optimizer converged to a **narrower, more twisted tip**: lower tip chord reduces
profile drag, and stronger washout pushes the spanwise lift distribution toward
the elliptic optimum.

## 6. Convergence history

In [7]:
history = result.convergence_history   # best objective value at each evaluation

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(-np.array(history), lw=1.5, color='steelblue')
ax.axhline(-result.objective_initial, color='grey', ls='--', lw=1, label=f'Baseline L/D = {-result.objective_initial:.1f}')
ax.set_xlabel('Evaluation number')
ax.set_ylabel('Best L/D')
ax.set_title('Convergence — scipy_de, glider wing')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('09_convergence.png', dpi=120)
plt.show()

print(f'Total evaluations: {result.n_evaluations}')
print(f'Best L/D: {-result.objective_optimal:.2f}')

Total evaluations: 82
Best L/D: 23.40


## 7. Sensitivity analysis

Which variable matters more? `sensitivity()` computes normalized finite-difference
gradients at the optimal point and ranks them by objective influence.

In [8]:
sens = problem.sensitivity(result.x_optimal)
print(sens.report())

Sensitivity Analysis — Objective Gradients (normalized, ranked)
  Evaluated at x=[0.0919, -5.9205]
  Objective value: -23.395
  Step size: 1.00e-04

  Objective:
   1. wings[0].xsecs[1].twist                                 dObj/dx=+0.177  |norm|=0.04479
   2. wings[0].xsecs[1].chord                                 dObj/dx=-3.088  |norm|=0.01213

  Constraint: aero.CL
   1. wings[0].xsecs[1].chord                                 dCon/dx=+1.398  |norm|=0.4183
   2. wings[0].xsecs[1].twist                                 dCon/dx=-0.02165  |norm|=0.4173


**Expected output:**
```
Sensitivity Analysis — Objective Gradients (normalized, ranked)
  Evaluated at x=[0.0985, -5.876]
  Objective value: -23.399
  Step size: 1.00e-04

  Objective:
   1. wings[0].xsecs[1].twist   dObj/dx=+0.1899  |norm|=0.04769
   2. wings[0].xsecs[1].chord   dObj/dx=-0.6293  |norm|=0.002648

  Constraint: aero.CL
   1. wings[0].xsecs[1].twist   dCon/dx=-0.02284  |norm|=0.4484
   2. wings[0].xsecs[1].chord   dCon/dx=+1.344    |norm|=0.4421
```

At the optimal point, **twist** drives L/D (positive gradient — more twist improves it),
while **chord** drives CL (positive gradient — a wider tip generates more lift). The
CL constraint is what limits the optimizer from pushing chord to its lower bound.

## 8. Baseline vs optimal comparison

Compare the full aerodynamic breakdown at baseline and optimal.

In [9]:
r_opt = problem.evaluate(result.x_optimal)['results']['aero']
r_bas = problem.evaluate(problem._x0_scaled())['results']['aero']

rows = [
    ('CL',           r_bas.CL,             r_opt.CL),
    ('CD',           r_bas.CD,             r_opt.CD),
    ('CDi (induced)',r_bas.CDi,            r_opt.CDi),
    ('CDp (profile)',r_bas.CD - r_bas.CDi, r_opt.CD - r_opt.CDi),
    ('L/D',          r_bas.CL_over_CD,     r_opt.CL_over_CD),
]

print(f'  {"Quantity":<20}  {"Baseline":>10}  {"Optimal":>10}  {"Change":>10}')
print('  ' + '-' * 56)
for name, v0, vopt in rows:
    pct = (vopt - v0) / abs(v0) * 100 if v0 else 0
    print(f'  {name:<20}  {v0:>10.4f}  {vopt:>10.4f}  {pct:>+9.1f}%')

print()
opt_chord = result.variables['wings[0].xsecs[1].chord'][1]
opt_twist = result.variables['wings[0].xsecs[1].twist'][1]
print(f'Optimal tip chord : {opt_chord:.3f} m  (taper ratio = {opt_chord/0.28:.2f})')
print(f'Optimal tip twist : {opt_twist:.1f} deg')

  Quantity                Baseline     Optimal      Change
  --------------------------------------------------------
  CL                        0.7623      0.7073       -7.2%
  CD                        0.0345      0.0302      -12.4%
  CDi (induced)             0.0198      0.0153      -22.9%
  CDp (profile)             0.0147      0.0150       +1.9%
  L/D                      22.1021     23.3953       +5.9%

Optimal tip chord : 0.092 m  (taper ratio = 0.33)
Optimal tip twist : -5.9 deg


**Expected output:**
```
  Quantity               Baseline     Optimal      Change
  --------------------------------------------------------
  CL                       0.7623      0.6992       -8.3%
  CD                       0.0345      0.0299      -13.3%
  CDi (induced)            0.0198      0.0151      -23.7%
  CDp (profile)            0.0147      0.0148       +0.7%
  L/D                     22.10       23.40         +5.9%

Optimal tip chord : 0.098 m  (taper ratio = 0.35)
Optimal tip twist : -5.9 deg
```

The gain is entirely from **reduced induced drag** (−24%). The optimizer reduces the
tip chord to lower the local wing loading and shapes the spanwise lift distribution
closer to the elliptic ideal. Profile drag is essentially unchanged — the area
reduction offsets the higher local lift coefficient at the narrower tip.

## 9. Wing planform visualisation

In [10]:
import copy
from aerisplane.mdo._paths import _unpack

# Reconstruct optimal aircraft to read geometry
opt_ac = _unpack(problem._baseline, problem._dvars, problem._pool_entries, result.x_optimal)
opt_wing  = opt_ac.wings[0]
base_wing = problem._baseline.wings[0]

def wing_planform(wing, flip=False):
    """Return (x, y) outline of one side of a symmetric wing in top-view."""
    xs = []
    for s in [0, 1]:
        xsec = wing.xsecs[s]
        x_le, y_le = xsec.xyz_le[0], xsec.xyz_le[1]
        c = xsec.chord
        xs.append((x_le, y_le, x_le + c, y_le))
    # leading edge outboard→inboard, then trailing edge inboard→outboard
    y  = [xs[1][1], xs[0][1]]
    le = [xs[1][0], xs[0][0]]
    te = [xs[1][2], xs[0][2]]
    ys  = y + y[::-1] + [y[0]]
    xs_ = le + te[::-1] + [le[0]]
    return xs_, ys

fig, ax = plt.subplots(figsize=(10, 4))

bx, by = wing_planform(base_wing)
ox, oy = wing_planform(opt_wing)

# Plot both half-wings (symmetric: mirror on y)
for sign in [1, -1]:
    ax.fill([x for x in bx], [sign * y for y in by],
            alpha=0.25, color='steelblue', label='Baseline' if sign == 1 else '')
    ax.plot([x for x in bx], [sign * y for y in by], color='steelblue', lw=1.5)
    ax.fill([x for x in ox], [sign * y for y in oy],
            alpha=0.25, color='tomato', label='Optimal' if sign == 1 else '')
    ax.plot([x for x in ox], [sign * y for y in oy], color='tomato', lw=1.5, ls='--')

ax.set_xlabel('x [m]  (chordwise)')
ax.set_ylabel('y [m]  (spanwise)')
ax.set_title('Wing planform: baseline vs optimal')
ax.set_aspect('equal')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('09_planform.png', dpi=120)
plt.show()

## Summary

| | Baseline | Optimal | Change |
|---|---|---|---|
| Tip chord | 0.140 m | ≈0.098 m | −30% |
| Tip twist | −2.0 deg | ≈−5.9 deg | stronger washout |
| L/D | 22.1 | 23.4 | **+5.9%** |
| CDi | 0.0198 | 0.0151 | −24% |

**Key take-aways:**

1. **`opti.variable()` replaces string paths** — design variables are declared at the
   point of geometry construction, so there's nothing to remember after the fact.

2. **`disciplines=["aero"]` is fast** — skipping structures, stability, and weights
   reduces each evaluation to a single aero call, keeping the total run under a minute.

3. **The optimizer recovers a near-elliptic distribution** — the reduced tip chord
   cuts induced drag (the dominant drag source at low speed) without measurably
   increasing profile drag.

4. **`simulate()` is your first check** — always run it before `optimize()` to
   confirm that constraint signs are correct and the baseline is physically reasonable.

**Next steps:** add `wings[0].xsecs[1].xyz_le[1]` (semispan) as a `DesignVar` string-path
variable alongside the Opti inline variables, and add a `Constraint('aero.CL', upper=0.9)`
upper bound to prevent the wing from stalling as span grows.